In [7]:
import pandas as pd         
import numpy as np  

import matplotlib.pyplot as plt   # plt es el alias convencional
import seaborn as sns             # sns es el alias convencional

# ── Librerías de Machine Learning (sklearn) 
from sklearn.preprocessing import StandardScaler  # Para escalar datos
from sklearn.cluster import KMeans                # Algoritmo K-Means
from sklearn.decomposition import PCA             # Análisis de componentes principales

In [8]:
# pd.read_csv() lee un archivo CSV y lo convierte en un DataFrame (tabla)
# Un DataFrame es la estructura principal de pandas: filas × columnas
df = pd.read_csv('viajaco_colombia.csv')

# .shape devuelve una tupla (filas, columnas)
print(f"📐 Dimensiones del dataset: {df.shape}")
print(f"   → {df.shape[0]} viajeros registrados")
print(f"   → {df.shape[1]} variables por viajero")

print("\n" + "─"*50)

# .head(n) muestra las primeras n filas — por defecto muestra 5
# Sirve para ver cómo lucen los datos sin cargar todo
print("\n📋 Primeras 5 filas del dataset:")
df.head()

📐 Dimensiones del dataset: (1500, 10)
   → 1500 viajeros registrados
   → 10 variables por viajero

──────────────────────────────────────────────────

📋 Primeras 5 filas del dataset:


,viajero_id,ciudad_origen,destino_preferido,tipo_viaje,canal_reserva,edad,gasto_promedio_cop,frecuencia_anual,noches_promedio,satisfaccion
0,VCO0001,Manizales,Eje Cafetero,Aventura,App móvil,18,345617,2.7,4.1,3.7
1,VCO0002,Bogotá,Medellín,Cultural,Reserva directa,46,1940263,1.2,2.2,5.0
2,VCO0003,Barranquilla,San Andrés,Descanso,Agencia tradicional,40,1309613,3.2,3.2,4.6
3,VCO0004,Barranquilla,Cartagena,Descanso,Reserva directa,33,932102,1.7,6.6,4.0
4,VCO0005,Bogotá,Santa Marta,Aventura,Recomendación,30,310989,2.0,4.3,3.0


In [9]:
# .dtypes muestra el tipo de dato de cada columna
# object = texto/categoría  |  int64 = entero  |  float64 = decimal
print("🗂️  Tipos de datos por columna:")
print(df.dtypes)

print("\n" + "─"*50)

# .isnull() crea una tabla booleana (True donde hay nulo)
# .sum() suma los True (en Python True == 1)
# Resultado: cuántos valores faltantes tiene cada columna
print("\n❓ Valores nulos por columna:")
nulos = df.isnull().sum()
print(nulos)
print(f"\n✅ Total de nulos en todo el dataset: {nulos.sum()}")

🗂️  Tipos de datos por columna:
viajero_id                str
ciudad_origen             str
destino_preferido         str
tipo_viaje                str
canal_reserva             str
edad                    int64
gasto_promedio_cop      int64
frecuencia_anual      float64
noches_promedio       float64
satisfaccion          float64
dtype: object

──────────────────────────────────────────────────

❓ Valores nulos por columna:
viajero_id            0
ciudad_origen         0
destino_preferido     0
tipo_viaje            0
canal_reserva         0
edad                  0
gasto_promedio_cop    0
frecuencia_anual      0
noches_promedio       0
satisfaccion          0
dtype: int64

✅ Total de nulos en todo el dataset: 0


In [10]:
print("📊 Estadísticas descriptivas de variables numéricas:")
df.describe().round(2)

📊 Estadísticas descriptivas de variables numéricas:


,edad,gasto_promedio_cop,frecuencia_anual,noches_promedio,satisfaccion
count,1500.00,1500.00,1500.00,1500.00,1500.00
mean,34.35,1189147.06,2.88,4.38,4.33
std,8.82,860673.49,1.55,1.51,0.45
min,18.00,150000.00,1.00,1.00,2.50
25%,27.00,381947.50,1.70,3.30,4.00
50%,34.00,1088448.00,2.40,4.20,4.40
75%,41.00,1751357.00,4.00,5.40,4.70
max,60.00,3310726.00,7.80,10.00,5.00


In [11]:
# .value_counts() cuenta cuántas veces aparece cada valor único en una columna
# Útil para variables categóricas (texto)
print("🗺️  Destinos más frecuentes:")
print(df['destino_preferido'].value_counts())

print("\n🎒 Tipos de viaje:")
print(df['tipo_viaje'].value_counts())

print("\n📱 Canal de reserva:")
# normalize=True convierte los conteos en proporciones (0 a 1)
# * 100 lo convierte a porcentaje
print((df['canal_reserva'].value_counts(normalize=True) * 100).round(1).astype(str) + '%')

🗺️  Destinos más frecuentes:
destino_preferido
Cartagena         372
Eje Cafetero      298
San Andrés        258
Santa Marta       219
Villa de Leyva    163
Amazonas          132
Medellín           58
Name: count, dtype: int64

🎒 Tipos de viaje:
tipo_viaje
Cultural        401
Descanso        366
Gastronómico    283
Aventura        276
Ecoturismo      174
Name: count, dtype: int64

📱 Canal de reserva:
canal_reserva
App móvil              36.0%
Reserva directa        26.3%
Agencia tradicional    25.4%
Recomendación          12.3%
Name: proportion, dtype: str


In [12]:
variables_numericas = ['edad', 'gasto_promedio_cop', 'frecuencia_anual', 'noches_promedio', 'satisfaccion']

X = df[variables_numericas].copy()

print(f" Shape de X (datos sin escalar): {X.shape}")
print("\nRangos origniales: ")
print(X.describe().loc[['mean', 'std', 'min', 'max']].round(2))

 Shape de X (datos sin escalar): (1500, 5)

Rangos origniales: 
       edad  gasto_promedio_cop  frecuencia_anual  noches_promedio  \
mean  34.35          1189147.06              2.88             4.38   
std    8.82           860673.49              1.55             1.51   
min   18.00           150000.00              1.00             1.00   
max   60.00          3310726.00              7.80            10.00   

      satisfaccion  
mean          4.33  
std           0.45  
min           2.50  
max           5.00  


In [18]:
#Creación del objeto standarScaler
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

print("Shape de X_scaled", X_scaled.shape)

print("\n Media por columna después del escalamiento: ")
for nombre, media in zip(variables_numericas, X_scaled.mean(axis=0)):
    print(f" {nombre:25s}: {media:.6f}")
    
    
print("\n STD por columna despues del escalado: ")
for nombre, std in zip(variables_numericas, X_scaled.std(axis=0)):
    print(f" {nombre:25s}: {std:.6f}")


Shape de X_scaled (1500, 5)

 Media por columna después del escalamiento: 
 edad                     : -0.000000
 gasto_promedio_cop       : 0.000000
 frecuencia_anual         : -0.000000
 noches_promedio          : -0.000000
 satisfaccion             : 0.000000

 STD por columna despues del escalado: 
 edad                     : 1.000000
 gasto_promedio_cop       : 1.000000
 frecuencia_anual         : 1.000000
 noches_promedio          : 1.000000
 satisfaccion             : 1.000000


In [20]:
#Calcular la inercia

inercias = []
valores_k = range(2,11)

for k in valores_k:
    
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    
    #fit()
    kmeans.fit(X_scaled)
    
    inercias.append(kmeans.inertia_)
    print(f"k={k:2d} -> Inercia: {kmeans.inertia_:>10,.2f}")
    

k= 2 -> Inercia:   4,028.70
k= 3 -> Inercia:   3,102.55
k= 4 -> Inercia:   2,629.23
k= 5 -> Inercia:   2,346.78
k= 6 -> Inercia:   2,173.35
k= 7 -> Inercia:   2,042.41
k= 8 -> Inercia:   1,913.35
k= 9 -> Inercia:   1,803.19
k=10 -> Inercia:   1,715.13
